In [2]:
import pandas as pd

In [5]:
master = pd.read_csv("labels.csv")

Index(['PATH_SERVER_NAME', 'NCI_PID', 'PATH_MASKED_IMG_ID', 'svs_delivered',
       'use_flag', 'duplicate_flag', 'ENRL_AGE', 'SCRN_HPV', 'wst_bx_site',
       'review_status', 'WST_EXPERT_DX', 'path_expert_img',
       'PATH_EXPERT_DX_OTHER', 'PATH_EXPERT_DX_NOTES',
       'PATH_LABEL_BIOPSY_TYPE', 'PATH_DICOM1', 'PATH_DICOM2', 'PATH_DICOM3',
       'PATH_DICOM4', 'PATH_DICOM5', 'PATH_DICOM6', 'PATH_DICOM7',
       'PATH_DICOM8', 'PATH_BATCH_NUM'],
      dtype='object')

In [6]:
master = master.rename(columns={
   'PATH_SERVER_NAME': 'country',
   'NCI_PID': 'patient_id',
   'ENRL_AGE': 'age',
   'PATH_MASKED_IMG_ID': 'image_id',
   'wst_bx_site': 'site_label',
   'path_expert_img': 'expert_label',
   'PATH_EXPERT_DX_OTHER': 'expert_label_other',
   'PATH_EXPERT_DX_NOTES': 'expert_label_notes',
    'SCRN_HPV': 'hpv_type'
})[[
      'image_id',
      'site_label',
      'expert_label',
      'expert_label_other',
      'expert_label_notes',
      'country',
      'patient_id',
      'age',
    'hpv_type',
      'svs_delivered',
]]

In [7]:
master['image_id'] = master['image_id'].str.replace('.SVS', '')

In [8]:
master['usable'] = 1
master['label'] = master['expert_label']

master.loc[
    master['expert_label'] == " ", 'usable'
] = 0

master.loc[
    (master['country'] == "Malawi") & (master['label'] == "Negative/Reactive"), 'usable'
] = 0

In [9]:
master.usable.value_counts()

usable
1    8170
0    7976
Name: count, dtype: int64

In [10]:
master.loc[
    master.expert_label == "Insufficient/Inadequate", 'label'
] = "insufficient"

master.loc[
    master.expert_label == "Atypia: Specify", 'label'
] = 'atypia'

master.loc[
    master.expert_label == "Other: Specify", 'label'
] = 'other'

master.loc[
    master.expert_label == 'CIN1', 'label'
] = 'low_grade'

master.loc[
    master.expert_label.isin(['CIN2','CIN3','AIS']), 'label'
] = 'high_grade'

master.loc[
    master.expert_label.isin([
        'Adenocarcinoma Invasive','Adenosquamous Carcinoma','Other Cancer: Specify','Squamous Invasive Carcinoma'
    ]) | 
    (
        (master.expert_label == "Other: Specify") & 
        (master.expert_label_other.str.contains("carcinoma", case=False)) &
        (~master.expert_label_other.str.contains("rule out", case=False))
    )  
    , 'label'
] = 'cancer'

master.loc[
    (master.expert_label == "Negative/Reactive") |
    (
        (master.expert_label == "Other: Specify") &
        (master.expert_label_other.str.contains("microglandular hyperplasia", case=False))
    ),
    'label'
] = 'normal'

In [11]:
master.loc[
    (master.label == "other") | (master.label == "atypia"),
    'usable'
] = 0

In [12]:
master['ready'] = False

from pathlib import Path

base = Path("/scratch/alpine/ataghinia@xsede.org/pv3")
for f in base.rglob("pt_files/*.pt"):
    image_id = f.parts[-1].split('.')[0]
    master.loc[master.image_id == image_id, 'ready'] = True

In [13]:
base = Path("/scratch/alpine/ataghinia@xsede.org/navyblue")
for f in base.rglob("pt_files/*.pt"):
    image_id = f.parts[-1].split('.')[0]
    master.loc[master.image_id == image_id, 'ready'] = True

In [14]:
pd.crosstab(master.usable, master.ready)

ready,False,True
usable,,
0,3147,4907
1,2934,5158


In [15]:
master[master.ready & master.usable].label.value_counts()

label
normal          2805
insufficient    1353
high_grade       643
low_grade        317
cancer            40
Name: count, dtype: int64

In [16]:
prepped = master[
    master.ready &
    master.usable == 1
][['patient_id', 'image_id', 'label', 'country', 'age']].rename(
    columns={
        'patient_id': 'case_id',
        'image_id': 'slide_id'
    }
)

In [19]:
print(f"N.patients: {prepped.case_id.nunique()}")

N.patients: 3260


In [20]:
print(f"N.slides: {len(prepped)}")

N.slides: 5158


In [27]:
prepped.to_csv("/projects/ataghinia@xsede.org/PAVE-Pathology/dataset_csv/pathology_full_subtyping.csv")

In [22]:
print("Descriptive Stats for Age")
prepped['age'].astype(int).describe()

Descriptive Stats for Age


count    5158.000000
mean       37.521714
std         6.468657
min         0.000000
25%        32.000000
50%        38.000000
75%        43.000000
max        50.000000
Name: age, dtype: float64

In [23]:
print("Slide Counts per Country")
prepped.country.value_counts()

Slide Counts per Country


country
El Salvador           1240
Cambodia              1064
Malawi                 804
Nigeria                513
Dominican Republic     492
Honduras               369
Tanzania               330
Brazil Brasilia        231
Eswatini               115
Name: count, dtype: int64

In [34]:
prepped.label.value_counts()

label
normal          2805
insufficient    1353
high_grade       643
low_grade        317
cancer            40
Name: count, dtype: int64